# Project Brazilian e-commerce by Olist

## Modeling Schema

<img src=..\docs\assets\data_schema.png width="900">


Source: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

## Data Dictionary

| Features | Description | Tables |
| :--- | :--- | :--- |
| customer_city | customer city name | customers |
| customer_id | key to the orders dataset. Each order has a unique customer_id. | customers, orders |
| customer_state | customer state | customers |
| customer_unique_id | unique identifier of a customer. | customers |
| customer_zip_code_prefix | first five digits of customer zip code | customers |
| freight_value | item freight value item (if an order has more than one item the freight value is splitted between items) | items |
| geolocation_city | city name | geolocation |
| geolocation_lat | latitude | geolocation |
| geolocation_lng | longitude | geolocation |
| geolocation_state | state | geolocation |
| geolocation_zip_code_prefix | first 5 digits of zip code | geolocation |
| order_approved_at | Shows the payment approval timestamp. | orders |
| order_delivered_carrier_date | Shows the order posting timestamp. When it was handled to the logistic partner. | orders |
| order_delivered_customer_date | Shows the actual order delivery date to the customer. | orders |
| order_estimated_delivery_date | Shows the estimated delivery date that was informed to customer at the purchase moment. | orders |
| order_id | order unique identifier | items, payments, reviews, orders |
| order_item_id | sequential number identifying number of items included in the same order | items |
| order_purchase_timestamp | Shows the purchase timestamp. | orders |
| order_status | Reference to the order status (delivered, shipped, etc). | orders |
| payment_installments | number of installments chosen by the customer. | payments |
| payment_sequential | a customer may pay an order with more than one payment method. If he does so, a sequence will be created to | payments |
| payment_type | method of payment chosen by the customer. | payments |
| payment_value | transaction value. | payments |
| price | item price | items |
| product_category_name | root category of product, in Portuguese. | products |
| > product_description_lenght | number of characters extracted from the product description. | products |
| product_height_cm | product height measured in centimeters. | products |
| product_id | product unique identifier | items, products |
| product_length_cm | product length measured in centimeters. | products |
| > product_name_lenght | number of characters extracted from the product name. | products |
| product_photos_qty | number of product published photos. | products |
| product_weight_g | product weight measured in grams. | products |
| product_width_cm | product width measured in centimeters. | products |
| review_answer_timestamp | Shows satisfaction survey answer timestamp. | reviews |
| review_comment_message | Comment message from the review left by the customer, in Portuguese. | reviews |
| review_comment_title | Comment title from the review left by the customer, in Portuguese. | reviews |
| review_creation_date | Shows the date in which the satisfaction survey was sent to the customer. | reviews |
| review_id | unique review identifier | reviews |
| review_score | Note ranging from 1 to 5 given by the customer on a satisfaction survey. | reviews |
| seller_city | seller city name | sellers |
| seller_id | seller unique identifier | items, sellers |
| seller_state | seller state | sellers |
| seller_zip_code_prefix | first 5 digits of seller zip code | sellers |
| shipping_limit_date | Shows the seller shipping limit date for handling the order over to the logistic partner. | items |


## Imports and configs

In [99]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [100]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from src.eda_utils import *
from src.data_processing import *
from src.plots import plot_numeric_distribution, plot_categorical_counts

In [101]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [102]:
DATE_COLS = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "review_creation_date",
    "review_answer_timestamp",
]

CAT_COLS = [
    "customer_city",
    "customer_state",
    "order_status",
    "payment_types",
    "product_category_name",
    "product_category_name_english",
    "seller_city",
    "seller_state",
]

ORDER_LEVEL_COLS = [
    "customer_id",
    "order_status",
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "customer_unique_id",
    "customer_zip_code_prefix",
    "customer_city",
    "customer_state",
    "payment_value_total",
    "payment_installments_max",
    "payment_sequential_max",
    "payment_types",
    "n_payment_methods",
    "review_id",
    "review_score",
    "review_comment_title",
    "review_comment_message",
    "review_creation_date",
    "review_answer_timestamp",
    "customer_lat",
    "customer_lng",
    "customer_geo_n_points",
]

ITEM_LEVEL_COLS = [
    "order_item_id",
    "product_id",
    "seller_id",
    "shipping_limit_date",
    "price",
    "freight_value",
    "product_category_name",
    "product_name_length",
    "product_description_length",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_category_name_english",
    "seller_zip_code_prefix",
    "seller_city",
    "seller_state",
    "seller_lat",
    "seller_lng",
    "seller_geo_n_points",
    "seller_customer_distance_km",
    "same_zip_prefix",
]

In [103]:
pd.reset_option("display.max_rows")

## Reading Data

In [104]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")
geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")
order_items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")
order_payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")
order_reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")
orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")
products = pd.read_csv("../data/raw/olist_products_dataset.csv")
sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

In [105]:
dfs = {
    "customers": customers,
    "geolocation": geolocation,
    "items": order_items,
    "payments": order_payments,
    "reviews": order_reviews,
    "orders": orders,
    "products": products,
    "sellers": sellers,
}

## Initial EDA

### Summary Statistics

In [106]:
for name, df in dfs.items():
    print(f"{name.upper()} {df.shape}:")
    display(dataframe_summary(df))

CUSTOMERS (99441, 5):


,dtype,total_null,percent_null,unique,count,top,freq,mean,std,25%,50%,75%,min,max,skew,kurtosis
feature,,,,,,,,,,,,,,,,
customer_city,str,0,0.0,4119,99441,sao paulo,15540,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,str,0,0.0,99441,99441,06b8999e2fba1a1fbc88172c00ba8bc7,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,str,0,0.0,27,99441,SP,41746,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,str,0,0.0,96096,99441,8d50f5eadf50201ccdcedfb9e2ac8455,17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,int64,0,0.0,14994,99441,22790,142,35137.474583,29797.938996,11347.0,24416.0,58900.0,1003.0,99990.0,0.779025,-0.788204


GEOLOCATION (1000163, 5):


,dtype,total_null,percent_null,unique,count,top,freq,mean,std,25%,50%,75%,min,max,skew,kurtosis
feature,,,,,,,,,,,,,,,,
geolocation_city,str,0,0.0,8011,1000163,sao paulo,135800,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_lat,float64,0,0.0,717360,1000163,-27.102099,314,-21.176153,5.715866,-23.603546,-22.919377,-19.979620,-36.605374,45.065933,1.565147,2.850097
geolocation_lng,float64,0,0.0,717613,1000163,-48.629613,314,-46.390541,4.269748,-48.573172,-46.637879,-43.767709,-101.466766,121.105394,-0.102417,4.727051
geolocation_state,str,0,0.0,27,1000163,SP,404268,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
geolocation_zip_code_prefix,int64,0,0.0,19015,1000163,24220,1146,36574.166466,30549.335710,11075.000000,26530.000000,63504.000000,1001.000000,99990.000000,0.694494,-0.941226


ITEMS (112650, 7):


...

,dtype,total_null,percent_null,unique,count,top,freq,mean,std,25%,50%,75%,min,max,skew,kurtosis
feature,,,,,,,,,,,,,,,,
customer_id,str,0,0.000000,99441,99441,9ef432eb6251297304e76186b10a928d,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_approved_at,str,160,0.160899,90733,99281,2018-02-27 04:31:10,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_carrier_date,str,1783,1.793023,81018,97658,2018-05-09 15:48:00,47,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_delivered_customer_date,str,2965,2.981668,95664,96476,2018-05-14 20:02:44,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_estimated_delivery_date,str,0,0.000000,459,99441,2017-12-20 00:00:00,522,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_id,str,0,0.000000,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,str,0,0.000000,98875,99441,2018-03-31 15:08:21,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_status,str,0,0.000000,8,99441,delivered,96478,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


PRODUCTS (32951, 9):


,dtype,total_null,percent_null,unique,count,top,freq,mean,std,25%,50%,75%,min,max,skew,kurtosis
feature,,,,,,,,,,,,,,,,
product_category_name,str,610,1.851234,73,32341,cama_mesa_banho,3029,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_description_lenght,float64,610,1.851234,2960,32341,404.0,94,771.495285,635.115225,339.0,595.0,972.0,4.0,3992.0,1.962093,4.828923
product_height_cm,float64,2,0.006070,102,32949,10.0,2548,16.937661,13.637554,8.0,13.0,21.0,2.0,105.0,2.140061,6.678619
product_id,str,0,0.000000,32951,32951,1e9e8ef04dbcff4541ed26657ea517e5,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_length_cm,float64,2,0.006070,99,32949,16.0,5520,30.815078,16.914458,18.0,25.0,38.0,7.0,105.0,1.750460,3.513618
product_name_lenght,float64,610,1.851234,66,32341,60.0,2182,48.476949,10.245741,42.0,51.0,57.0,5.0,76.0,-0.903222,0.192563
product_photos_qty,float64,610,1.851234,19,32341,1.0,16489,2.188986,1.736766,1.0,1.0,3.0,1.0,20.0,2.193409,7.263534
product_weight_g,float64,2,0.006070,2204,32949,200.0,2084,2276.472488,4282.038731,300.0,700.0,1900.0,0.0,40425.0,3.604860,15.133565
product_width_cm,float64,2,0.006070,95,32949,11.0,3718,23.196728,12.079047,15.0,20.0,30.0,6.0,118.0,1.670971,4.073126


SELLERS (3095, 4):


,dtype,total_null,percent_null,unique,count,top,freq,mean,std,25%,50%,75%,min,max,skew,kurtosis
feature,,,,,,,,,,,,,,,,
seller_city,str,0,0.0,611,3095,sao paulo,694,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_id,str,0,0.0,3095,3095,3442f8959a84dea7ee197c632cb2df15,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_state,str,0,0.0,23,3095,SP,1849,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_zip_code_prefix,int64,0,0.0,2246,3095,14940,49,32291.059451,32713.45383,7093.5,14940.0,64552.5,1001.0,99730.0,0.916092,-0.858264


### Observations

- O order_id está duplicado em orders (deveria ser único)? `Não`
- review_id duplicado ou múltiplos review_id para o mesmo order_id em order_reviews? `Sim, será tratado no merge dos dados` 
- Unicidade de order_id + order_item_id em order_items? `Sim, não há duplicidade das chaves compostas`
- product_id duplicado em products, seller_id duplicado em sellers? `Não`
- review_score sempre entre 1 e 5 em order_reviews? `Sim`
- Valores negativos/zero em price, freight_value (order_items), payment_value (order_payments), product_weight_g (products)? `Não há valores negativos, porém as colunas freight_value, payment_installments, payment_value e product_weight_g estão com valor mínimo zero, o que pode fazer sentido no caso de freight_value, mas não para os demais.`

In [107]:
dfs["payments"].payment_installments.unique()

array([ 8,  1,  2,  3,  6,  5,  4, 10,  7, 12,  9, 13, 15, 24, 11, 18, 14,
       20, 21, 17, 22,  0, 16, 23])

In [108]:
# Verifica se a combinação order_id + order_item_id é única
duplicates = dfs["items"].duplicated(subset=["order_id", "order_item_id"]).sum()
print(f"Linhas duplicadas na chave composta: {duplicates}")

Linhas duplicadas na chave composta: 0


In [109]:
strange_values = {
    "items": ["freight_value"],
    "payments": ["payment_installments", "payment_value"],
    "products": ["product_weight_g"],
}

for table, columns in strange_values.items():
    for col in columns:
        mask = dfs[table][col] == 0
        print(f"coluna {col} zerada")
        display(dfs[table][mask])
        if table == "items":
            display(dfs[table][mask].value_counts("seller_id"))
            # display(show_multi_item_orders(dfs[table][mask], required_cols = dfs[table].columns))

coluna freight_value zerada


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
...,...,...,...,...,...,...,...
111094,fc698f330ec7fb74859071cc6cb29772,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-04-25 02:31:57,53.9,0.0
111497,fd4907109f6bac23f07064af84bec02d,1,7a10781637204d8d10485c71a6108a2e,4869f7a5dfa277a7dca6462dcf3b52b2,2018-04-30 11:31:32,219.0,0.0
111649,fd95e4b85ebbb81853d4a6be3d61432b,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,2018-05-04 11:10:31,106.9,0.0
112182,fee19a0dc7358b6962a611cecf6a37b4,1,f1c7f353075ce59d8a6f3cf58f419c9c,37be5a7c751166fbc5f8ccba4119e043,2017-09-07 22:06:31,195.0,0.0


seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
c826c40d7b19f62a09e2d7c5e7295ee2      2
bc2ac6b95e1accce9858528ee566c17e      1
cc419e0650a3c5ba77189a1882b7556a      1
8581055ce74af1daba164fdbd55a40de      1
Name: count, dtype: int64

coluna payment_installments zerada


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


coluna payment_value zerada


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


coluna product_weight_g zerada


,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
9769,81781c0fed9fe1ad6e8c81fca1e1cb08,cama_mesa_banho,51.0,529.0,1.0,0.0,30.0,25.0,30.0
13683,8038040ee2a71048d4bdbbdc985b69ab,cama_mesa_banho,48.0,528.0,1.0,0.0,30.0,25.0,30.0
14997,36ba42dd187055e1fbe943b2d11430ca,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0
32079,e673e90efa65a5409ff4196c038bb5af,cama_mesa_banho,53.0,528.0,1.0,0.0,30.0,25.0,30.0


- Frete zerado pode ser considerado valor legitimo e por se tratar de vendedores especificos demonstra um padrão, vale a pena investigar mais sobre, após o merge das tabelas e checar se há algum padrão associado a categoria, localização e preço.
- payment_installments zerado é provavel erro, pois o número mínimo de parcelas de pagamento deveria ser 1, porém tratando-se apenas de dois pedidos tem impacto mínimo no projeto.
- payment_value zerado pode representar erro, principalmente para payment_type not_defined vale a pena verificar novamente após o merge.
- product_weight_g zerado é um valor impossivel, o que também indica erro.

O tratamento desses erros e dados nulos serão feitos após o merge das tabelas, pois com mais contexto é possivel tomar uma decisão mais acertiva com relação a descartar ou preencher os valores.

In [110]:
mask = dfs["orders"]["order_delivered_customer_date"].isna()
dfs["orders"][mask].value_counts("order_status")

order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2
Name: count, dtype: int64

- A feature order_delivered_customer_date é essencial para o objetivo de prever possivel atraso e oder_status delivered é o que mais faz sentido neste contexto, então ter apenas 8 nulos nesta categoria é um bom sinal. 

### Cardinality Checking

1. order_items — esperado 1:N

In [111]:
# Conta quantos itens (linhas) cada order_id tem
items_per_order = dfs["items"].groupby("order_id").size()

# Distribuição: quantos pedidos têm 1 item, 2 itens, 3 itens, etc.
print(items_per_order.value_counts().sort_index())

# Estatística resumida
print(items_per_order.describe())

1     88863
2      7516
3      1322
4       505
5       204
6       198
7        22
8         8
9         3
10        8
11        4
12        5
13        1
14        2
15        2
20        2
21        1
Name: count, dtype: int64
count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64


2. order_payments — checar se existe 1:N

In [112]:
# Conta quantas linhas de pagamento cada order_id tem
payments_per_order = dfs["payments"].groupby("order_id").size()

print(payments_per_order.value_counts().sort_index())
print(payments_per_order.describe())

# Opcional: olhar exemplos de pedidos com múltiplas linhas de pagamento
multi_payment_orders = payments_per_order[payments_per_order > 1].index
dfs["payments"][dfs["payments"]["order_id"].isin(multi_payment_orders)].sort_values(
    "order_id"
).head(20)

1     96479
2      2382
3       301
4       108
5        52
6        36
7        28
8        11
9         9
10        5
11        8
12        8
13        3
14        2
15        2
19        2
21        1
22        1
26        1
29        1
Name: count, dtype: int64
count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64


,order_id,payment_sequential,payment_type,payment_installments,payment_value
80856,0016dfedd97fc2950e388d2971d718c7,2,voucher,1,17.92
89575,0016dfedd97fc2950e388d2971d718c7,1,credit_card,5,52.63
20036,002f19a65a2ddd70a090297872e6d64e,1,voucher,1,44.11
98894,002f19a65a2ddd70a090297872e6d64e,2,voucher,1,33.18
10244,0071ee2429bc1efdc43aa3e073a5290e,1,voucher,1,100.00
30155,0071ee2429bc1efdc43aa3e073a5290e,2,voucher,1,92.44
285,009ac365164f8e06f59d18a08045f6c4,5,voucher,1,8.75
15298,009ac365164f8e06f59d18a08045f6c4,6,voucher,1,4.17
16053,009ac365164f8e06f59d18a08045f6c4,1,credit_card,1,0.88
16459,009ac365164f8e06f59d18a08045f6c4,2,voucher,1,4.50


3. order_reviews — checar se é ~1:1

In [113]:
reviews_per_order = dfs["reviews"].groupby("order_id")["review_id"].nunique()

multi_review_orders = (reviews_per_order > 1).sum()
print(
    f"Pedidos com múltiplos reviews: {multi_review_orders} ({multi_review_orders / reviews_per_order.shape[0]:.2%})"
)

multi_review_orders = reviews_per_order[reviews_per_order > 1].index
df_multi_review = dfs["reviews"][
    dfs["reviews"]["order_id"].isin(multi_review_orders)
].sort_values("order_id")

display(
    df_multi_review[
        [
            "order_id",
            "review_id",
            "review_score",
            "review_creation_date",
            "review_answer_timestamp",
        ]
    ].head(20)
)

Pedidos com múltiplos reviews: 547 (0.55%)


,order_id,review_id,review_score,review_creation_date,review_answer_timestamp
22423,0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-25 00:00:00,2017-08-29 21:45:57
25612,0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-29 00:00:00,2017-08-30 01:59:12
22779,013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-22 00:00:00,2018-02-23 12:12:30
68633,013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-04 00:00:00,2018-03-05 17:02:00
854,0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2017-12-30 00:00:00,2018-01-02 10:54:06
83224,0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2017-12-30 00:00:00,2018-01-02 10:54:47
17582,02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,2018-03-29 00:00:00,2018-03-30 03:16:19
89888,02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,2018-03-21 00:00:00,2018-03-22 01:32:08
37911,029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,2017-07-14 00:00:00,2017-07-17 13:58:06
55137,029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,2017-07-19 00:00:00,2017-07-20 12:06:11


- Deveria haver apenas um review por pedido, mas o usuário pode ter atualizada a sua review e gerado outra, a estratégia que faz mais sentido é manter apenas o ultimo review de acordo com a variável review_answer_timestamp.

## Merge

Para dar prosseguimento as analises irei juntar todas as tabelas exceto geolocation (por enquanto), contudo surgem algumas perguntas que devem ser respondidas antes do merge: 
- `"O que representa uma linha no meu dataset unificado?"` ->  após breve analise cheguei a conclusão que o nivel mais granular que ainda faz sentido é o de item do pedido (order_items), pois produto, vendedor, frete e preço variam por item e não por um pedido inteiro.
- `"Como as demais tabelas se relacionam com o item do pedido?"` -> Isso pode ser respondido pela relação de cardinalidade:

| Tabela | Relação com "item do pedido" |
|---|---|
| `orders` | 1 pedido → N itens (então 1:m em relação ao grão) |
| `order_items` | **é o próprio grão** |
| `products` | N itens podem repetir o mesmo produto (m:1) |
| `sellers` | N itens podem repetir o mesmo vendedor (m:1) |
| `customers` | 1 pedido = 1 cliente, mas N itens do mesmo pedido repetem o cliente (m:1 em relação ao grão) |
| `payments` | 1 pedido pode ter N pagamentos — **não bate direto com o grão "item"** |
| `reviews` | 1 pedido normalmente tem 1 review, raramente mais — **também não bate diretamente** |

payments e reviews são relações "por pedido", não "por item". Se eu simplesmente desse merge direto delas contra o dataset já expandido por item, qualquer pedido com 2 itens e 2 pagamentos viraria 4 linhas (2×2) — um produto cartesiano que não representa nada real, só um artefato do merge.

Dito isto, a ordem do merge fica:

```
orders → customers → order_items → products → sellers → payments (agregado) → reviews (dedup)
```

- **`orders + customers` primeiro**: relação 1:1 real (cada pedido tem exatamente um cliente), então é seguro e não altera a contagem de linhas. Fiz isso cedo porque é "barato" e não introduz risco.
- **`+ order_items`**: aqui, e só aqui, o dataset "explode" de fato — de propósito, porque é essa a granularidade que eu quero.
- **`+ products` e `+ sellers`**: ambos são m:1 em relação ao grão atual (vários itens podem apontar pro mesmo produto/vendedor), então o merge não altera a contagem de linhas — só adiciona colunas.
- **`+ payments`**: como *não* é m:1 em relação ao grão, eu não faço merge direto. Primeiro **agrego** (`groupby('order_id')`) para colapsar a granularidade de payments para "por pedido", e só depois faço o merge — agora sim é m:1 seguro.
- **`+ reviews`**: mesmo raciocínio — dedupliquei por `order_id` antes, transformando numa relação segura.



In [114]:
# Roda o script de merge das tabela, e trata o problemas encontrados anteriormente
%run ../src/data_processing.py

2026-10-28 19:41:35,765 [INFO] Carregando customers de ..\data\raw\olist_customers_dataset.csv
2026-10-28 19:41:35,988 [INFO] Carregando orders de ..\data\raw\olist_orders_dataset.csv
2026-10-28 19:41:36,876 [INFO] Carregando order_items de ..\data\raw\olist_order_items_dataset.csv
2026-10-28 19:41:37,406 [INFO] Carregando payments de ..\data\raw\olist_order_payments_dataset.csv
2026-10-28 19:41:37,655 [INFO] Carregando reviews de ..\data\raw\olist_order_reviews_dataset.csv
2026-10-28 19:41:38,298 [INFO] Carregando products de ..\data\raw\olist_products_dataset.csv
2026-10-28 19:41:38,492 [INFO] Carregando sellers de ..\data\raw\olist_sellers_dataset.csv
2026-10-28 19:41:38,503 [INFO] Carregando geolocation de ..\data\raw\olist_geolocation_dataset.csv
2026-10-28 19:41:39,437 [INFO] Carregando category_translation de ..\data\raw\product_category_name_translation.csv
2026-10-28 19:41:39,807 [INFO] orders + customers -> (99441, 12)
2026-10-28 19:41:40,203 [INFO] + order_items -> (112650, 

## Integrity Checks

In [115]:
df_processed = pd.read_parquet("../data/processed/olist_processed.parquet")
df_processed.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value_total,payment_installments_max,payment_types,n_payment_methods,payment_sequential_max,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_lat,customer_lng,customer_geo_n_points,seller_lat,seller_lng,seller_geo_n_points
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40,268,4,500,19,8,13,housewares,09350,maua,SP,38.71,1,credit_card | voucher,2,3,a54f0611adc9ed256b57ede6b6eb5114,4,<NA>,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,-23.576170,-46.587276,24,-23.681180,-46.444127,207
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29,178,1,400,19,13,19,perfumery,31570,belo horizonte,SP,141.46,1,boleto,1,1,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,-12.126651,-45.008162,19,-19.807013,-43.980966,71
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46,232,1,420,24,19,21,auto,14840,guariba,SP,179.12,3,credit_card,1,1,e73b67b67587f7644d5bd1a52deb1b01,5,<NA>,<NA>,2018-08-18,2018-08-22 19:07:58,-16.744472,-48.514624,25,-21.364020,-48.228831,170


In [116]:
df_processed.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'product_category_name_english', 'seller_zip_code_prefix',
       'seller_city', 'seller_state', 'payment_value_total',
       'payment_installments_max', 'payment_types', 'n_payment_methods',
       'payment_sequential_max', 'review_id', 'review_score',
       'review_comment_title', 'review_comment_message',
       'review_creation_date', 'review_answer_timestamp', 'customer_

In [117]:
df_processed.rename(
    columns={
        "product_description_lenght": "product_description_length",
        "product_name_lenght": "product_name_length",
    },
    inplace=True,
)

### Summary Statistics (item)

In [118]:
# save_table_as_image(
#     dataframe_summary(df_processed), "../reports/figures/table_preview.png"
# )

In [119]:
dataframe_summary(df_processed)

,dtype,total_null,percent_null,unique,count,top,freq,mean,std,25%,50%,75%,min,max,skew,kurtosis
feature,,,,,,,,,,,,,,,,
customer_city,string,0,0.000000,4110,112650,sao paulo,17808,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_geo_n_points,Int32,303,0.268975,483,112347,44,763,153.059494,153.167960,53.000000,103.000000,196.000000,1,1146,2.303723,7.275247
customer_id,string,0,0.000000,98666,112650,fc3d1daec319d62d49bfb5e1f83123e9,21,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_lat,float64,303,0.268975,14815,112347,-23.014295,154,-21.237194,5.561168,-23.590926,-22.930150,-20.182699,-33.690972,3.844931,1.623272,2.920821
customer_lng,float64,303,0.268975,14814,112347,-43.464132,154,-46.202997,4.034195,-48.109214,-46.634834,-43.635422,-72.670987,-34.799347,-0.001185,2.009423
customer_state,category,0,0.000000,27,112650,SP,47449,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,string,0,0.000000,95420,112650,c8460e4251689ba205045f3ea17884a1,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,string,0,0.000000,14976,112650,22790,154,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freight_value,float64,0,0.000000,6999,112650,15.1,3707,19.990320,15.806405,13.080000,16.260000,21.150000,0.0,409.68,5.639870,59.788253


### Observations (items)

Os dois objetivos principais do projeto estão relacionados ao grão em nivel de pedido, então será necessario colapsar e agregar os dados por pedido, porém antes disto é necessario fazer correções e analises no nivel de item para não contaminar os valores na hora de agregá-los.

1. **`shipping_limit_date` com máximo em 2020-04-09.** O dataset cobre 2016-2018 (compras até 2018-09), então essa data está fora do período. Vale investigar as linhas afetadas.

2. **`product_weight_g` e `freight_value` com mínimo 0.** Como visto anteriormente peso zero é impossível e o frete pode ser válido.

3. 1.603 nulos idênticos em `product_category_name`, `product_name_length`, `product_description_length` e `product_photos_qty`, o que sugere as mesmas linhas com metadados ausentes.

4. `product_category_name_english` tem 1.627 nulos, ou seja, **24 a mais** que a categoria em português. São categorias que não estão na tabela de tradução, e dá para traduzir manualmente.

5. Só 3 linhas sem pagamento (provavelmente um único pedido com 3 itens).

6. **`payment_installments_max` com mínimo 0.** Um pagamento com zero parcelas é suspeito.

Obs: **112.650 linhas para 98.666 pedidos.** É o grão de item, como esperado. O `order_id` mais frequente aparece 21 vezes e `order_item_id` chega a 21, então há um pedido com 21 itens, que pesa 21 vezes mais que um pedido comum, então é necessario ter cautela ao colapsar para o nível de pedido. Outro ponto que vale resaltar é que tinhamos 99441 pedidos na tabela **ORDERS**, porém perdemos 775 pedidos devido alguns deles não terem referencia na tabela **ITEMS**.

#### shipping_limit_date

In [120]:
df_processed.sort_values(by=["shipping_limit_date"], ascending=False)[DATE_COLS]

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,review_creation_date,review_answer_timestamp
97799,2017-05-23 22:28:36,2017-05-24 22:35:08,2017-05-29 02:03:28,2017-06-09 13:35:54,2017-10-11,2020-04-09 22:35:08,2017-06-10,2017-06-12 13:24:30
97800,2017-05-23 22:28:36,2017-05-24 22:35:08,2017-05-29 02:03:28,2017-06-09 13:35:54,2017-10-11,2020-04-09 22:35:08,2017-06-10,2017-06-12 13:24:30
40799,2017-03-16 02:30:51,2017-03-16 02:30:51,NaT,NaT,2017-08-08,2020-02-05 03:30:51,2017-08-11,2017-08-11 16:52:36
62658,2017-03-14 19:23:22,2017-03-14 19:23:22,2017-03-16 14:31:15,NaT,2017-08-04,2020-02-03 20:23:22,2017-05-17,2017-05-17 22:04:19
63193,2018-08-25 20:59:18,2018-08-25 21:10:15,2018-08-28 15:05:00,2018-08-30 16:47:53,2018-09-21,2018-09-18 21:10:15,2018-08-31,2018-09-02 22:51:48
...,...,...,...,...,...,...,...,...
7602,2016-10-03 22:31:31,2016-10-04 10:19:23,2016-10-08 10:34:01,2016-10-14 16:08:00,2016-11-23,2016-10-08 10:34:01,2016-10-19,2016-10-22 00:23:57
34802,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,2016-09-19 23:11:33,2016-10-06,2016-10-07 18:32:28
34803,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,2016-09-19 23:11:33,2016-10-06,2016-10-07 18:32:28
34804,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,2016-09-19 23:11:33,2016-10-06,2016-10-07 18:32:28


In [121]:
for col in DATE_COLS:
    mask = df_processed[col].dt.year >= 2019
    print(f"{col} - {len(df_processed[mask])}")

order_purchase_timestamp - 0
order_approved_at - 0
order_delivered_carrier_date - 0
order_delivered_customer_date - 0
order_estimated_delivery_date - 0
shipping_limit_date - 4
review_creation_date - 0
review_answer_timestamp - 0


- É um aparente erro, pois não faz sentido a compra ser aprovada em 2017 e ter prazo de entrega para 2020, irei inserir valor nulo visto que não temos uma data confiável

In [122]:
mask = df_processed["shipping_limit_date"].dt.year >= 2019
df_processed.loc[mask, "shipping_limit_date"] = pd.NaT
df_processed.loc[mask, DATE_COLS]

,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,shipping_limit_date,review_creation_date,review_answer_timestamp
40799,2017-03-16 02:30:51,2017-03-16 02:30:51,NaT,NaT,2017-08-08,NaT,2017-08-11,2017-08-11 16:52:36
62658,2017-03-14 19:23:22,2017-03-14 19:23:22,2017-03-16 14:31:15,NaT,2017-08-04,NaT,2017-05-17,2017-05-17 22:04:19
97799,2017-05-23 22:28:36,2017-05-24 22:35:08,2017-05-29 02:03:28,2017-06-09 13:35:54,2017-10-11,NaT,2017-06-10,2017-06-12 13:24:30
97800,2017-05-23 22:28:36,2017-05-24 22:35:08,2017-05-29 02:03:28,2017-06-09 13:35:54,2017-10-11,NaT,2017-06-10,2017-06-12 13:24:30


#### product_weight_g

In [123]:
mask = df_processed["product_weight_g"] <= 0
df_processed.loc[mask]["product_id"].value_counts()

product_id
e673e90efa65a5409ff4196c038bb5af    4
36ba42dd187055e1fbe943b2d11430ca    2
8038040ee2a71048d4bdbbdc985b69ab    1
81781c0fed9fe1ad6e8c81fca1e1cb08    1
Name: count, dtype: int64[pyarrow]

In [124]:
df_processed.loc[mask, "product_weight_g"] = pd.NA

In [125]:
df_processed["product_weight_g"].isna().sum()

np.int64(26)

- Saimos de 18 para 26

#### freight_value

In [126]:
mask = df_processed["freight_value"] <= 0
for col in [
    "product_id",
    "product_category_name",
    "price",
    "seller_id",
    "seller_state",
    "seller_city",
]:
    display(df_processed[mask][col].value_counts()[:5])

product_id
53b36df67ebb7c41585e8d54d6772e08    187
aca2eb7d00ea1a7b8ebd4e68314663af     98
422879e10f46682990de24d770e7f83d     56
7a10781637204d8d10485c71a6108a2e     27
f1c7f353075ce59d8a6f3cf58f419c9c      9
Name: count, dtype: int64[pyarrow]

product_category_name
relogios_presentes    214
moveis_decoracao       98
ferramentas_jardim     56
cama_mesa_banho         9
livros_importados       1
Name: count, dtype: int64[pyarrow]

price
99.9     120
69.9      98
53.9      56
110.0     27
199.0     23
Name: count, dtype: int64

seller_id
7d13fca15225358621be4086e1eb0964    158
955fee9216a65b617aa5c0531780ce60     99
1f50f920176fa81dab994f9023523100     56
4869f7a5dfa277a7dca6462dcf3b52b2     56
37be5a7c751166fbc5f8ccba4119e043      9
Name: count, dtype: int64[pyarrow]

seller_state
SP    383
AC      0
AM      0
BA      0
CE      0
Name: count, dtype: int64

seller_city
ribeirao preto           158
sao paulo                108
sao jose do rio preto     56
guariba                   56
guarulhos                  3
Name: count, dtype: int64[pyarrow]

- Parece haver um padrão entre os itens de frete gratis o que reforça o valor legitmo e não um erro, por isso vamos manter como esta.

#### null values

In [127]:
df_processed[["product_category_name", "product_name_length", "product_description_length", "product_photos_qty"]].isna().all(axis="columns").sum()

np.int64(1603)

In [128]:
df_processed[["product_weight_g", "product_width_cm", "product_length_cm", "product_height_cm"]].isna().all(axis="columns").sum()

np.int64(18)

In [129]:
mask = df_processed[["payment_installments_max", "payment_types", "payment_value_total", "payment_sequential_max", "n_payment_methods"]].isna().all(axis="columns")
df_processed[mask]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value_total,payment_installments_max,payment_types,n_payment_methods,payment_sequential_max,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_lat,customer_lng,customer_geo_n_points,seller_lat,seller_lng,seller_geo_n_points
34802,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,1,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34,1036,1,1000,16,16,16,health_beauty,81810,curitiba,PR,NaN,<NA>,<NA>,<NA>,<NA>,6916ca4502d6d3bfd39818759d55d536,1,<NA>,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28,-20.585396,-47.863156,345,-25.507138,-49.276529,141
34803,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,2,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34,1036,1,1000,16,16,16,health_beauty,81810,curitiba,PR,NaN,<NA>,<NA>,<NA>,<NA>,6916ca4502d6d3bfd39818759d55d536,1,<NA>,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28,-20.585396,-47.863156,345,-25.507138,-49.276529,141
34804,bfbd0f9bdef84302105ad712db648a6c,86dc2ffce2dfff336de2f386a786e574,delivered,2016-09-15 12:16:38,2016-09-15 12:16:38,2016-11-07 17:11:53,2016-11-09 07:47:38,2016-10-04,830d5b7aaa3b6f1e9ad63703bec97d23,14600,sao joaquim da barra,SP,3,5a6b04657a4c5ee34285d1e4619a96b4,ecccfa2bb93b34a3bf033cc5d1dcdc69,2016-09-19 23:11:33,44.99,2.83,beleza_saude,34,1036,1,1000,16,16,16,health_beauty,81810,curitiba,PR,NaN,<NA>,<NA>,<NA>,<NA>,6916ca4502d6d3bfd39818759d55d536,1,<NA>,nao recebi o produto e nem resposta da empresa,2016-10-06,2016-10-07 18:32:28,-20.585396,-47.863156,345,-25.507138,-49.276529,141


- De fato 1603 valores nulos pertencem as mesmas linhas nas quatro variáveis `product_category_name`, `product_name_length`, `product_description_length` e `product_photos_qty`, assim como os 18 valores nulos para `product_weight_g`, `product_width_cm`, `product_length_cm` e `product_height_cm`.
- Os 3 valores nulos realcionados as variáveis de pagamento realmente pertecem a um mesmo pedido

#### payment_installments_max

In [130]:
mask = df_processed["payment_installments_max"] <= 0
df_processed[mask]

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value_total,payment_installments_max,payment_types,n_payment_methods,payment_sequential_max,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_lat,customer_lng,customer_geo_n_points,seller_lat,seller_lng,seller_geo_n_points
72139,744bade1fcf9ff3f31d860ace076d422,5e5794daaa13f73e2f1cdb4114529843,delivered,2018-04-22 11:34:42,2018-04-24 19:04:46,2018-04-24 03:14:34,2018-04-27 20:55:28,2018-05-16,f54cea27c80dc09bfe07b1cf1e01b845,05263,sao paulo,SP,1,0cf573090c66bb30ac5e53c82bdb0403,7202e2ba20579a9bd1acb29e61fe71f6,2018-04-26 12:31:06,45.90,12.79,telefonia,26,108,1,128,15,8,13,telephony,14401,franca,SP,58.69,0,credit_card,1,2,c56b4b7042f417e70d3dc5340deb23fe,5,<NA>,<NA>,2018-04-28,2018-04-29 17:43:50,-23.433009,-46.581423,5,-20.528048,-47.388279,271
75127,1a57108394169c0b47d8f876acc9ba2d,48ebb06cf56dba9d009230cc751bb195,delivered,2018-05-15 16:25:14,2018-05-15 16:36:52,2018-05-17 12:37:00,2018-05-24 15:45:41,2018-06-06,9925e1d7dff0d807355599dee04830ab,44001,feira de santana,BA,1,db35a562fb6ba63e19fa42a15349dc04,282f23a9769b2690c5dda22e316f9941,2018-05-18 16:31:54,41.69,23.28,utilidades_domesticas,56,235,1,500,23,11,23,housewares,31573,belo horizonte,MG,129.94,0,credit_card,1,2,bc5dca6c957c328b1cbff760b0841fd2,5,<NA>,<NA>,2018-05-25,2018-05-26 00:24:28,-12.255816,-38.960887,134,-19.811157,-43.989872,90
75128,1a57108394169c0b47d8f876acc9ba2d,48ebb06cf56dba9d009230cc751bb195,delivered,2018-05-15 16:25:14,2018-05-15 16:36:52,2018-05-17 12:37:00,2018-05-24 15:45:41,2018-06-06,9925e1d7dff0d807355599dee04830ab,44001,feira de santana,BA,2,db35a562fb6ba63e19fa42a15349dc04,282f23a9769b2690c5dda22e316f9941,2018-05-18 16:31:54,41.69,23.28,utilidades_domesticas,56,235,1,500,23,11,23,housewares,31573,belo horizonte,MG,129.94,0,credit_card,1,2,bc5dca6c957c328b1cbff760b0841fd2,5,<NA>,<NA>,2018-05-25,2018-05-26 00:24:28,-12.255816,-38.960887,134,-19.811157,-43.989872,90


In [131]:
df_processed[mask]["order_id"].value_counts()

order_id
1a57108394169c0b47d8f876acc9ba2d    2
744bade1fcf9ff3f31d860ace076d422    1
Name: count, dtype: int64[pyarrow]

In [132]:
df_processed.loc[mask, "payment_installments_max"] = pd.NA

#### Payment Consistency

In [133]:
multi_item_df = show_multi_item_orders(df_processed)
multi_item_df

,order_id,price,freight_value,payment_value_total
0,0008288aa423d2a3f00fcb17cd7d8719,49.90,13.37,126.54
1,0008288aa423d2a3f00fcb17cd7d8719,49.90,13.37,126.54
2,00143d0f86d6fbd9f9b38ab440ac16f5,21.33,15.10,109.29
3,00143d0f86d6fbd9f9b38ab440ac16f5,21.33,15.10,109.29
4,00143d0f86d6fbd9f9b38ab440ac16f5,21.33,15.10,109.29
...,...,...,...,...
23782,fff8287bbae429a99bb7e8c21d151c41,180.00,48.14,456.28
23783,fffb9224b6fc7c43ebb0904318b10b5f,55.00,34.19,356.76
23784,fffb9224b6fc7c43ebb0904318b10b5f,55.00,34.19,356.76
23785,fffb9224b6fc7c43ebb0904318b10b5f,55.00,34.19,356.76


In [134]:
consistency_df = check_payment_consistency(df_processed)

# resumo rápido
print(consistency_df["is_consistent"].value_counts())
print(f"Total de pedidos inconsistentes: {(~consistency_df['is_consistent']).sum()}")

# ver só os inconsistentes, ordenados pela maior diferença
inconsistent = consistency_df[~consistency_df["is_consistent"]].sort_values(
    "difference", ascending=False
)
inconsistent

is_consistent
True     98388
False      278
Name: count, dtype: int64
Total de pedidos inconsistentes: 278


,order_id,payment_types,price_freight_sum,payment_value_total,difference,is_consistent
79538,ce6d150fb29ada17d2082f4847107665,credit_card,1403.66,1586.47,182.81,False
42515,6e5fe7366a2e1bfbf3257dba0af1267f,credit_card,287.91,406.92,119.01,False
43434,70b742795bc441e94a44a084b6d9ce7a,credit_card,466.93,578.82,111.89,False
58752,996c7e73600ad3723e8627ab7bef81e4,credit_card,587.90,664.43,76.53,False
43437,70b7e94ea46d3e8b5bc12a50186edaf0,credit_card,213.15,274.84,61.69,False
...,...,...,...,...,...,...
11060,1cb61aaa63135f552055cf1932adf3be,credit_card,293.64,293.66,0.02,False
86128,df13cbaf3230c62a4b582317936e8a39,credit_card,297.04,297.02,0.02,False
39788,674eb13687ba86abeaee12ece9ce0309,credit_card,148.00,147.98,0.02,False
11435,1daecbe2a13ab0881c418e978f1e08c1,credit_card,241.19,241.17,0.02,False


In [135]:
inconsistent.payment_types.value_counts()

payment_types
credit_card              259
debit_card                 7
credit_card | voucher      7
boleto                     4
Name: count, dtype: int64[pyarrow]

- Apenas uma pequena quantidade de pagamentos apresenta inconsistencia, o que significa que o merge com a tabela payments foi bem sucedido e os valores discrepantes podem ocorrer por sujeira ou outros fatores.  

#### product_category_name

In [136]:
mask = (~df_processed['product_category_name'].isna() & df_processed['product_category_name_english'].isna())
df_processed.loc[mask, ['product_category_name', 'product_category_name_english']]

,product_category_name,product_category_name_english
1041,portateis_cozinha_e_preparadores_de_alimentos,<NA>
2502,pc_gamer,<NA>
8425,portateis_cozinha_e_preparadores_de_alimentos,<NA>
10179,portateis_cozinha_e_preparadores_de_alimentos,<NA>
12664,portateis_cozinha_e_preparadores_de_alimentos,<NA>
13120,pc_gamer,<NA>
19462,portateis_cozinha_e_preparadores_de_alimentos,<NA>
19550,pc_gamer,<NA>
30236,portateis_cozinha_e_preparadores_de_alimentos,<NA>
30237,portateis_cozinha_e_preparadores_de_alimentos,<NA>


In [137]:
mask = df_processed["product_category_name"] == "portateis_cozinha_e_preparadores_de_alimentos"
df_processed.loc[mask, 'product_category_name_english'] = "kitchen_portables_and_food_preparators"
mask = df_processed["product_category_name"] == "pc_gamer"
df_processed.loc[mask, 'product_category_name_english'] = "pc_gamer"

#### Distance between customer and seller

In [138]:
df_processed = add_seller_customer_distance(
    df_processed,
    customer_lat_col= "customer_lat",
    customer_lng_col= "customer_lng",
    seller_lat_col= "seller_lat",
    seller_lng_col= "seller_lng",
)

df_processed.head(3)

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english,seller_zip_code_prefix,seller_city,seller_state,payment_value_total,payment_installments_max,payment_types,n_payment_methods,payment_sequential_max,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_lat,customer_lng,customer_geo_n_points,seller_lat,seller_lng,seller_geo_n_points,seller_customer_distance_km,same_zip_prefix
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,03149,sao paulo,SP,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72,utilidades_domesticas,40,268,4,500,19,8,13,housewares,09350,maua,SP,38.71,1,credit_card | voucher,2,3,a54f0611adc9ed256b57ede6b6eb5114,4,<NA>,"Não testei o produto ainda, mas ele veio correto e em boas condições. Apenas a caixa que veio bem amassada e danificada, o que ficará chato, pois se trata de um presente.",2017-10-11,2017-10-12 03:43:48,-23.576170,-46.587276,24,-23.681180,-46.444127,207,18.681737,0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76,perfumaria,29,178,1,400,19,13,19,perfumery,31570,belo horizonte,SP,141.46,1,boleto,1,1,8d5266042046a06655c8db133d120ba5,4,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,-12.126651,-45.008162,19,-19.807013,-43.980966,71,861.036556,0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22,automotivo,46,232,1,420,24,19,21,auto,14840,guariba,SP,179.12,3,credit_card,1,1,e73b67b67587f7644d5bd1a52deb1b01,5,<NA>,<NA>,2018-08-18,2018-08-22 19:07:58,-16.744472,-48.514624,25,-21.364020,-48.228831,170,514.547851,0


- Como a distância entre cliente e vendedor pode ser interessante é necessario calcular antes de colapsar para o nivel de pedido, pois um pedido pode ter mais de um vendedor então esse nivel de detalhamento será perdido. 

### Order Aggregation 

In [139]:
def check_order_level_consistency(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    """Conta quantos pedidos têm mais de um valor distinto em cada coluna de pedido."""
    nunique = df.groupby("order_id")[cols].nunique(dropna=False)
    return (nunique > 1).sum()


display(check_order_level_consistency(df_processed, ORDER_LEVEL_COLS))

display(check_order_level_consistency(df_processed, ITEM_LEVEL_COLS))

customer_id                      0
order_status                     0
order_purchase_timestamp         0
order_approved_at                0
order_delivered_carrier_date     0
order_delivered_customer_date    0
order_estimated_delivery_date    0
customer_unique_id               0
customer_zip_code_prefix         0
customer_city                    0
customer_state                   0
payment_value_total              0
payment_installments_max         0
payment_sequential_max           0
payment_types                    0
n_payment_methods                0
review_id                        0
review_score                     0
review_comment_title             0
review_comment_message           0
review_creation_date             0
review_answer_timestamp          0
customer_lat                     0
customer_lng                     0
customer_geo_n_points            0
dtype: int64

order_item_id                    9803
product_id                       3236
seller_id                        1278
shipping_limit_date               336
price                            2391
freight_value                    2030
product_category_name             786
product_name_length              2688
product_description_length       2824
product_photos_qty               1298
product_weight_g                 2820
product_length_cm                1934
product_height_cm                1963
product_width_cm                 2032
product_category_name_english     786
seller_zip_code_prefix           1126
seller_city                      1026
seller_state                      510
seller_lat                       1126
seller_lng                       1126
seller_geo_n_points              1123
seller_customer_distance_km      1121
same_zip_prefix                     0
dtype: int64

In [ ]:
df_agg = collapse_item_level_to_order_level(df_processed, order_level_cols=ORDER_LEVEL_COLS)

df_agg.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,payment_value_total,payment_installments_max,payment_sequential_max,payment_types,n_payment_methods,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,customer_lat,customer_lng,customer_geo_n_points,n_items,n_distinct_products,n_sellers,n_seller_states,seller_states,seller_cities,max_shipping_limit_date,min_shipping_limit_date,total_price,max_item_price,avg_item_price,total_freight,main_category,n_categories,categories,mean_name_length,mean_description_length,min_photos_qty,total_weight_g,max_weight_g,total_volume_cm3,max_item_volume_cm3,max_seller_distance_km,mean_seller_distance_km,n_sellers_missing_distance,min_seller_geo_n_points,max_seller_geo_n_points
0,00010242fe8c5a6d1ba2dd792cb16214,3ce436f183e68e07877b285a838db11a,delivered,2017-09-13 08:59:02,2017-09-13 09:45:35,2017-09-19 18:34:16,2017-09-20 23:43:48,2017-09-29,871766c5855e863f6eccc05f988b23cb,28013,campos dos goytacazes,RJ,72.19,2,1,credit_card,1,97ca439bc427b48bc1cd7177abe71365,5,<NA>,"Perfeito, produto entregue antes do combinado.",2017-09-21,2017-09-22 10:57:03,-21.762829,-41.310605,149,1,1,1,1,SP,volta redonda,2017-09-19 09:45:35,2017-09-19 09:45:35,58.90,58.90,58.90,13.29,cool_stuff,1,cool_stuff,58.0,598.0,4,650,650,3528,3528,301.229621,301.229621,0,59,59
1,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,eb28e67c4c0b83846050ddfb8a35d051,15775,santa fe do sul,SP,259.83,3,1,credit_card,1,7b07bacd811c4117b742569b04ce3580,4,<NA>,<NA>,2017-05-13,2017-05-15 11:34:13,-20.205737,-50.926924,367,1,1,1,1,SP,sao paulo,2017-05-03 11:05:13,2017-05-03 11:05:13,239.90,239.90,239.90,19.93,pet_shop,1,pet_shop,56.0,239.0,2,30000,30000,60000,60000,588.404882,588.404882,0,39,39
2,000229ec398224ef6ca0657da4fc703e,6489ae5e4333f3693df5ad4372dab6d3,delivered,2018-01-14 14:33:31,2018-01-14 14:48:30,2018-01-16 12:36:48,2018-01-22 13:19:16,2018-02-05,3818d81c6709e39d06b2738a8d3a2474,35661,para de minas,MG,216.87,5,1,credit_card,1,0c5b33dea94867d1ac402749e5438e8b,5,<NA>,Chegou antes do prazo previsto e o produto surpreendeu pela qualidade. Muito satisfatório.,2018-01-23,2018-01-23 16:06:31,-19.870383,-44.594355,224,1,1,1,1,MG,borda da mata,2018-01-18 14:48:30,2018-01-18 14:48:30,199.00,199.00,199.00,17.87,furniture_decor,1,furniture_decor,59.0,695.0,2,3050,3050,14157,14157,312.835455,312.835455,0,71,71
3,00024acbcdf0a6daa1e931b038114c75,d4eb9395c8c0431ee92fce09860c5a06,delivered,2018-08-08 10:00:35,2018-08-08 10:10:18,2018-08-10 13:28:00,2018-08-14 13:32:39,2018-08-20,af861d436cfc08b2c2ddefd0ba074622,12952,atibaia,SP,25.78,2,1,credit_card,1,f4028d019cb58564807486a6aaf33817,4,<NA>,<NA>,2018-08-15,2018-08-15 16:39:01,-23.104346,-46.595589,27,1,1,1,1,SP,franca,2018-08-15 10:10:18,2018-08-15 10:10:18,12.99,12.99,12.99,12.79,perfumery,1,perfumery,42.0,480.0,1,200,200,2400,2400,295.032888,295.032888,0,438,438
4,00042b26cf59d7ce69dfabb4e55b4fd9,58dbd0b2d70206bf40e62cd34e84d795,delivered,2017-02-04 13:57:51,2017-02-04 14:10:13,2017-02-16 09:46:09,2017-03-01 16:42:31,2017-03-17,64b576fb70d441e8f1b2d7d446e483c5,13226,varzea paulista,SP,218.04,3,1,credit_card,1,940144190dcba6351888cafa43f3a3a5,5,<NA>,Gostei pois veio no prazo determinado .,2017-03-02,2017-03-03 10:54:59,-23.245512,-46.825182,4,1,1,1,1,PR,loanda,2017-02-13 13:57:51,2017-02-13 13:57:51,199.90,199.90,199.90,18.14,garden_tools,1,garden_tools,59.0,409.0,1,3750,3750,42000,42000,646.469437,646.469437,0,119,119


In [92]:
df_agg = df_agg[sorted(df_agg.columns)]
df_agg.shape

(98666, 53)

### Summary Statistics (order)

In [ ]:
# save_table_as_image(
#     dataframe_summary(df_agg), "../reports/figures/df_agg_summary.png"
# )

In [ ]:
# dataframe_summary(df_agg).to_excel("../reports/figures/df_agg_summary.xlsx")

In [141]:
dataframe_summary(df_agg)

,dtype,total_null,percent_null,unique,count,top,freq,mean,std,25%,50%,75%,min,max,skew,kurtosis
feature,,,,,,,,,,,,,,,,
avg_item_price,float64,0,0.000000,7125,98666,59.9,2004,125.919255,190.985636,41.990000,79.000000,139.900000,0.85,6735.0,7.761884,115.388547
categories,string,1389,1.407780,318,97277,bed_bath_table,9219,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_city,string,0,0.000000,4110,98666,sao paulo,15402,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_geo_n_points,Int32,275,0.278718,483,98391,84,659,152.105396,152.357987,53.000000,103.000000,194.000000,1,1146,2.308502,7.323235
customer_id,string,0,0.000000,98666,98666,3ce436f183e68e07877b285a838db11a,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_lat,float64,275,0.278718,14815,98391,-23.014295,142,-21.192006,5.602082,-23.588425,-22.924930,-20.137901,-33.690972,3.844931,1.612436,2.831738
customer_lng,float64,275,0.278718,14814,98391,-43.464132,142,-46.174362,4.053652,-48.104199,-46.630866,-43.597068,-72.670987,-34.799347,0.003283,1.984464
customer_state,category,0,0.000000,27,98666,SP,41375,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,string,0,0.000000,95420,98666,8d50f5eadf50201ccdcedfb9e2ac8455,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [161]:
s = 1 if " | ".join(["z"]) else None
s

1

In [163]:
df_agg["payment_types"].unique()

<ArrowStringArray>
[             'credit_card',                   'boleto',
    'credit_card | voucher',               'debit_card',
                  'voucher', 'credit_card | debit_card',
                       <NA>]
Length: 7, dtype: string

In [157]:
mask = df_agg["order_delivered_customer_date"].isna()
df_agg.loc[mask, "order_status"].value_counts()

order_status
shipped        1106
canceled        455
invoiced        312
processing      301
delivered         8
unavailable       6
approved          2
created           0
Name: count, dtype: int64

In [149]:
df_agg["order_status"].unique()

['delivered', 'shipped', 'canceled', 'invoiced', 'processing', 'approved', 'unavailable']
Categories (8, str): ['approved', 'canceled', 'created', 'delivered', 'invoiced', 'processing', 'shipped', 'unavailable']

In [97]:
df_agg['seller_states'].value_counts()

seller_states
SP              69754
MG               7763
PR               7507
RJ               4262
SC               3575
                ...  
PE | PR             1
GO | PR             1
GO | SP             1
AC                  1
MG | SC | SP        1
Name: count, Length: 68, dtype: int64[pyarrow]

In [98]:
df_agg["n_distinct_products"].value_counts()

n_distinct_products
1    95430
2     2846
3      298
4       70
6       10
5        8
7        3
8        1
Name: count, dtype: Int64

### Observations (orders)

Saimos agora de um dataset com 112.650 itens para um com 98.666 pedidos o que é o esperado visto que 

1. `order_delivered_customer_date` tem 2.454 nulos, quase igual às 2.453 linhas que não estão como `delivered` (112.650 − 110.197). É necessario cruzar com `order_status` para confirmar que os nulos coincidem com pedidos não entregues.

2. `order_delivered_carrier_date` (1.194) e `order_approved_at` (15) seguem a mesma lógica.

3. **A tabela `orders` original tem 99.441 pedidos.** Portanto o `inner` em `merge_order_items` descartou cerca de 775 pedidos sem itens. O `how="inner"`, fez a categoria `created` de `order_status` desaparecer e maior partes dos pedidos `unavailable` também não foram encontrados.

4. **`payment_value_total` está inflado.** A média é 180,28, contra 140,64 de `price + freight_value` (120,65 + 19,99). Em pedidos com vários itens, cada linha repete o pagamento total do pedido, ou seja, essa coluna no grão de item não deve ser somada.

5. **`customer_id` e `order_id` têm 98.666 únicos cada**, e `customer_unique_id` tem 95.420. Ou seja, há pouca recompra, algo que pode aparecer na análise de negócio.

6. Colunas `review_*` com 942 nulos são pedidos sem avaliação.

11. `review_comment_title` (88%) e `review_comment_message` (58%) são campos opcionais. Provavelmente o mais útil é uma flag como `has_comment`.

#### Distribuições

- **Assimetria forte** em `price` (skew 7,9 e curtose 120, máximo 6.735 contra mediana de 74,99), em `payment_value_total` (skew 13,9) e em `product_weight_g`. Para o XGBoost isso não exige transformação, mas para gráficos e modelos lineares a escala logarítmica ajuda.
- **`review_score` concentrado em 5**: 56,5% das linhas (63.157 de 111.708), com skew de −1,27. Se o alvo for "avaliação ruim", a classe positiva será minoritária.
- **97,8% dos pedidos estão como `delivered`.** Para o modelo de atraso, só esses têm data de entrega para calcular o alvo.

#### outros


**Validação de agregação**
- Conferir que `payment_value_total` (após `groupby("first")`) bate com a soma esperada por pedido, sem duplicação por item
- Conferir que `n_items`, `n_distinct_sellers`, `avg_product_weight` etc. batem com o esperado por amostragem manual de alguns `order_id`

### Dates

In [53]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "review_creation_date",
    "review_answer_timestamp",
    "max_shipping_limit_date",
    "min_shipping_limit_date"
]

#### NaT Values

In [49]:
mask = df_agg.order_status != "delivered"
mask.sum()

np.int64(2188)

In [50]:
df_agg.order_status.value_counts()

order_status
delivered      96478
shipped         1106
canceled         461
invoiced         312
processing       301
unavailable        6
approved           2
created            0
Name: count, dtype: int64

Datas nulas de pedidos não entregues

In [54]:
df_agg.loc[mask, date_cols].isna().sum()

order_purchase_timestamp            0
order_approved_at                   0
order_delivered_carrier_date     1007
order_delivered_customer_date    2182
order_estimated_delivery_date       0
review_creation_date              103
review_answer_timestamp           103
max_shipping_limit_date             2
min_shipping_limit_date             2
dtype: int64

Datas nulas de pedidos entregues

In [55]:
df_agg.loc[~mask, date_cols].isna().sum()

order_purchase_timestamp           0
order_approved_at                 14
order_delivered_carrier_date       2
order_delivered_customer_date      8
order_estimated_delivery_date      0
review_creation_date             646
review_answer_timestamp          646
max_shipping_limit_date            1
min_shipping_limit_date            1
dtype: int64

- Quase todos os nulos da coluna `order_delivered_customer_date` não são da categoria "delivered", o que é esperado visto que não há como um pedido ter data de entrega sem ter sido entregue. Em contrapartida há apenas 8 items sem data de entrega pertencente a categoria "delivered", assim como a coluna `order_delivered_carrier_date` e `order_approved_at` com apenas 2 e 15 respectivamente, um valor baixo se comparado aos ~100 mil pedidos, o que é otimo, pois para o objetivo de prever se o pedido ira atrasar o recorte ideal é justamente os pedidos da categoria "delivered" de order_status.  

#### Date Integrity

In [75]:
DATE_ORDER_RULES = [
    ("order_purchase_timestamp", "order_approved_at"),
    ("order_purchase_timestamp", "order_delivered_carrier_date"),
    ("order_purchase_timestamp", "order_delivered_customer_date"),
    ("order_purchase_timestamp", "order_estimated_delivery_date"),
    ("order_purchase_timestamp", "shipping_limit_date"),
    ("order_approved_at", "order_delivered_carrier_date"),
    ("order_approved_at", "order_estimated_delivery_date"),
    ("order_approved_at", "order_delivered_customer_date"),
    ("order_delivered_carrier_date", "order_delivered_customer_date"),
]


def check_date_integrity(df: pd.DataFrame) -> pd.DataFrame:
    """Verifica se as datas do pedido seguem a ordem cronológica esperada.

    Retorna um resumo com uma linha por regra, contendo quantas linhas
    foram checadas e quantas violaram a ordem.
    """
    results = []

    for earlier, later in DATE_ORDER_RULES:
        # Só compara linhas em que as duas datas existem (nulos estruturais são ignorados)
        both_present = df[earlier].notna() & df[later].notna()
        violations = both_present & (df[later] < df[earlier])

        checked_rows = int(both_present.sum())
        violation_count = int(violations.sum())

        results.append(
            {
                "rule": f"{earlier} <= {later}",
                "checked_rows": checked_rows,
                "violations": violation_count,
                "violation_pct": round(100 * violation_count / max(checked_rows, 1), 2),
            }
        )

    return pd.DataFrame(results)

In [76]:
summary = check_date_integrity(df_processed[~mask])
display(summary)

,rule,checked_rows,violations,violation_pct
0,order_purchase_timestamp <= order_approved_at,112629,0,0.00
1,order_purchase_timestamp <= order_delivered_carrier_date,111450,187,0.17
2,order_purchase_timestamp <= order_delivered_customer_date,110190,0,0.00
3,order_purchase_timestamp <= order_estimated_delivery_date,112644,0,0.00
4,order_purchase_timestamp <= shipping_limit_date,112644,0,0.00
5,order_approved_at <= order_delivered_carrier_date,111435,1550,1.39
6,order_approved_at <= order_estimated_delivery_date,112629,9,0.01
7,order_approved_at <= order_delivered_customer_date,110175,69,0.06
8,order_delivered_carrier_date <= order_delivered_customer_date,110189,50,0.05


- A variável order_approved_at parece apresentar erros de registro, pois cronologicamente um pedido só deve ser enviado após a aprovação do pagamento e as violações vão contra essa logica.
- O cutoff temporal do projeto é a variável `order_approved_at` o que evita vazamento de dados, isto é descrito no item 8 em `doc/business_understanding.md`, então as demais variáveis relacionadas ao periodo posterior não serão utilizadas como preditoras, o que torna a maioria das violações não tão graves para o nosso objetivo, porém as violações `order_approved_at <= order_estimated_delivery_date` e `order_approved_at <= order_delivered_customer_date` irão prejudicar o calculo da variavel target "delayed".

In [ ]:
mask = (
    df_processed["order_approved_at"].notna()
    & df_processed["order_delivered_customer_date"].notna()
) & (df_processed["order_approved_at"] >= df_processed["order_delivered_customer_date"])
df_processed[mask]["order_status"].value_counts()

order_status
delivered      69
approved        0
canceled        0
created         0
invoiced        0
processing      0
shipped         0
unavailable     0
Name: count, dtype: int64

In [41]:
# Isola as linhas inconsistentes e calcula a magnitude da violação
mask = df_processed["order_approved_at"] > df_processed["order_delivered_customer_date"]
violations = df_processed.loc[mask].copy()
violations["gap_hours"] = (
    violations["order_approved_at"] - violations["order_delivered_customer_date"]
).dt.total_seconds() / 3600

display(violations["gap_hours"].describe())
display(violations["order_status"].value_counts())

count     69.000000
mean      42.784110
std       45.442543
min        0.088611
25%        3.452222
50%       27.654167
75%       67.141389
max      167.751667
Name: gap_hours, dtype: float64

order_status
delivered      69
approved        0
canceled        0
created         0
invoiced        0
processing      0
shipped         0
unavailable     0
Name: count, dtype: int64

### Order Status

In [ ]:
plot_categorical_counts(df_processed, "order_status", multicolor=True)

In [ ]:
# Pedidos sem data de entrega
customer_date_null = df_processed["order_delivered_customer_date"].isna()
plot_categorical_counts(
    df_processed[customer_date_null], "order_status", multicolor=True
)

# Inserção e tratamento

In [ ]:
mask = df_processed["payment_installments_max"] <= 0
display(df_processed[mask])
mask = df_processed["product_weight_g"] <= 0
display(df_processed[mask])

In [ ]:
# Cria coluna com pedidos atrasados preservando os valores nulos
df_processed["delayed"] = (
    df_processed["order_delivered_customer_date"].dt.normalize()
    > df_processed["order_estimated_delivery_date"].dt.normalize()
).mask(df_processed["order_delivered_customer_date"].isna())
df_processed["delayed"].sum()

In [ ]:
plot_categorical_counts(
    df_processed[df_processed["delayed"] == True], "order_status", multicolor=True
)

In [ ]:
plot_categorical_counts(df_processed, "review_score", multicolor=True)

In [ ]:
df_processed["payment_types"].value_counts(dropna=False)

In [ ]:
df_processed.groupby("order_id")["review_id"].count().sort_values(ascending=True)

In [ ]:
verify_items = show_multi_item_orders(
    df_processed,
    required_cols=[
        "order_id",
        "product_id",
        "product_category_name",
        "product_weight_g",
        "price",
        "freight_value",
        "payment_value_total",
        "review_id",
        "review_comment_message",
        "review_score",
        "review_creation_date",
        "seller_id",
        "seller_city",
        "order_status",
        "delayed",
    ],
)

verify_items.head(40)